
# PIF engine stress tests

This notebook starts after expand_pif.ipynb has already produced reusable artifacts. The goal is to avoid re-reading or rebuilding the same ENPG, mortality, and AAF inputs in this second pass.

Working order:

1. Locate __andres_control from the current Positron session.
2. Find and load the latest dated artifacts of each type.
3. Expose the saved exposure and AAF objects produced by expand_pif.ipynb.
4. Define PIF counterfactual scenarios for consumption volume and HED prevalence.
5. Run optional smoke tests before any full Monte Carlo run.
6. Combine PIF outputs with cached YPLL.
7. Run assertion checks before exporting any policy-counterfactual result.


In [5]:

#| label: pif2-setup
#| results: "hold"

.t0 <- Sys.time()
# Keep the second-stage notebook gentle in an interactive Positron session.
# Set this to TRUE only when you intentionally want to remove objects from memory.
pif2_clear_workspace <- FALSE
if (isTRUE(pif2_clear_workspace)) {
  rm(list = ls())
  invisible(gc())
}
# The notebook is verbose by default so users can see which files are selected,
# why they are selected, and which objects become available downstream.
pif2_verbose <- TRUE
# Use explicit namespace calls instead of library() so the notebook remains clear
# about where each helper comes from.
pif2_required_packages <- c("dplyr", "tidyr", "purrr", "tibble", "readxl", "knitr")
pif2_missing_packages <- pif2_required_packages[
  !vapply(pif2_required_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(pif2_missing_packages) > 0) {
  stop("Missing required package(s): ", paste(pif2_missing_packages, collapse = ", "))
}
message(sprintf(
  "pif2-setup elapsed minutes: %.2f",
  as.numeric(difftime(Sys.time(), .t0, units = "mins"))
))


pif2-setup elapsed minutes: 0.00



## Latest-artifact helper

The helper below is intentionally verbose. It searches by file type or pattern, extracts dates from filenames when possible, and selects the newest candidate by embedded date first and modification time second. If no filename date exists, it falls back to modification time.


In [6]:

#| label: pif2-latest-artifact-helpers
#| results: "hold"

.t0 <- Sys.time()

pif2_elapsed_min <- function(.start = .t0) {
  as.numeric(difftime(Sys.time(), .start, units = "mins"))
}
pif2_message <- function(..., verbose = pif2_verbose) {
  if (isTRUE(verbose)) {
    message(sprintf(...))
  }
}
pif2_find_control_dir <- function(verbose = pif2_verbose) {
  # Positron can run a notebook from the project root, from __andres_control,
  # or from another working directory. This helper tries all common locations
  # and returns the directory that contains the PIF/AAF engine files.
  candidates <- unique(c(
    normalizePath(".", winslash = "/", mustWork = FALSE),
    normalizePath(file.path(".", "__andres_control"), winslash = "/", mustWork = FALSE),
    normalizePath(file.path("..", "__andres_control"), winslash = "/", mustWork = FALSE)
  ))
  hits <- candidates[
    file.exists(file.path(candidates, "aaf_unified.R")) &
      file.exists(file.path(candidates, "rr_registry_adam.R"))
  ]
  if (!length(hits)) {
    stop("Could not find __andres_control with aaf_unified.R and rr_registry_adam.R from getwd(): ", getwd())
  }
  control_dir <- normalizePath(hits[[1L]], winslash = "/", mustWork = TRUE)
  pif2_message("[control-dir] Using: %s", control_dir, verbose = verbose)
  control_dir
}
# Define project root
pif2_project_root <- function(control_dir) {
  normalizePath(file.path(control_dir, ".."), winslash = "/", mustWork = TRUE)
}
pif2_extract_embedded_date <- function(file_name, date_regex = "(\\d{8})") {
  # Return a Date parsed from YYYYMMDD inside the filename. If the filename does
  # not carry an 8-digit date, return NA so the caller can fall back to mtime.
  hit <- regmatches(file_name, regexpr(date_regex, file_name, perl = TRUE))
  if (!length(hit) || is.na(hit) || !nzchar(hit)) {
    return(as.Date(NA))
  }
  as.Date(hit, format = "%Y%m%d")
}
pif2_latest_matching_file <- function(directory,
                                      pattern,
                                      date_regex = "(\\d{8})",
                                      recursive = FALSE,
                                      prefer_embedded_date = TRUE,
                                      verbose = pif2_verbose) {
  # This is the generic selector. It is deliberately not tied to one artifact
  # name, so it can be reused for any future output type from expand_pif.ipynb.
  directory <- normalizePath(directory, winslash = "/", mustWork = TRUE)
  pif2_message("[latest-file] Directory: %s", directory, verbose = verbose)
  pif2_message("[latest-file] Pattern: %s", pattern, verbose = verbose)
  files <- list.files(
    path = directory,
    pattern = pattern,
    full.names = TRUE,
    recursive = recursive,
    ignore.case = FALSE
  )
  if (!length(files)) {
    stop("No files matched pattern '", pattern, "' under ", directory)
  }
  info <- file.info(files)
  candidates <- tibble::tibble(
    path = normalizePath(files, winslash = "/", mustWork = TRUE),
    file_name = basename(files),
    embedded_date = as.Date(vapply(basename(files), pif2_extract_embedded_date, as.Date(NA), date_regex = date_regex)),
    modified_time = info$mtime,
    size_bytes = info$size
  )
  has_dates <- any(!is.na(candidates$embedded_date))
  if (isTRUE(prefer_embedded_date) && has_dates) {
    candidates <- candidates |>
      dplyr::arrange(dplyr::desc(.data$embedded_date), dplyr::desc(.data$modified_time), dplyr::desc(.data$size_bytes))
    pif2_message("[latest-file] Selection rule: embedded YYYYMMDD date, then modified time.", verbose = verbose)
  } else {
    candidates <- candidates |>
      dplyr::arrange(dplyr::desc(.data$modified_time), dplyr::desc(.data$size_bytes))
    pif2_message("[latest-file] Selection rule: modified time fallback.", verbose = verbose)
  }

  pif2_message("[latest-file] Candidate count: %s", nrow(candidates), verbose = verbose)
  pif2_message("[latest-file] Selected: %s", candidates$file_name[[1L]], verbose = verbose)
  attr(candidates$path[[1L]], "candidates") <- candidates
  candidates$path[[1L]]
}
pif2_latest_dated_file <- function(directory,
                                   stem,
                                   extension = "rds",
                                   verbose = pif2_verbose) {
  # Convenience wrapper for artifacts named like stem_YYYYMMDD.extension.
  escaped_stem <- gsub("([.|()\\^{}+$*?]|\\[|\\])", "\\\\\\1", stem, perl = TRUE)
  escaped_extension <- gsub("([.|()\\^{}+$*?]|\\[|\\])", "\\\\\\1", extension, perl = TRUE)
  pattern <- paste0("^", escaped_stem, "_\\d{8}\\.", escaped_extension, "$")
  pif2_latest_matching_file(directory, pattern = pattern, verbose = verbose)
}
pif2_read_latest_artifact <- function(directory,
                                      pattern = NULL,
                                      stem = NULL,
                                      extension = "rds",
                                      reader = NULL,
                                      verbose = pif2_verbose) {
  # Find the latest file and read it with the right parser. The return value keeps
  # the selected path in attr(x, "path") so later cells can report provenance.
  selected_path <- if (!is.null(pattern)) {
    pif2_latest_matching_file(directory, pattern = pattern, verbose = verbose)
  } else {
    pif2_latest_dated_file(directory, stem = stem, extension = extension, verbose = verbose)
  }
  if (is.null(reader)) {
    ext <- tolower(tools::file_ext(selected_path))
    reader <- switch(
      ext,
      rds = base::readRDS,
      xlsx = readxl::read_xlsx,
      csv = utils::read.csv,
      stop("No default reader for extension: ", ext)
    )
  }
  pif2_message("[read-artifact] Reading: %s", selected_path, verbose = verbose)
  out <- reader(selected_path)
  attr(out, "path") <- selected_path
  out
}
message(sprintf(
  "pif2-latest-artifact-helpers elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

pif2-latest-artifact-helpers elapsed minutes: 0.00



## Load saved outputs from expand_pif.ipynb

This section loads the latest saved bundles and exposes the objects needed by the PIF engine. It sources engine code, but it does not rebuild ENPG exposure objects or mortality tables from raw files.


In [7]:
#| label: pif2-load-latest-artifacts
#| results: "hold"

.t0 <- Sys.time()
# define directoriess
pif2_control_dir <- pif2_find_control_dir()
pif2_root_dir <- pif2_project_root(pif2_control_dir)

# Source code helpers only. These are lightweight compared with rebuilding ENPG,
# mortality, and AAF objects from raw data.
source(file.path(pif2_control_dir, "aaf_unified.R"))
source(file.path(pif2_control_dir, "rr_registry_adam.R"))
pif2_message("[engine] Loaded aaf_unified.R and rr_registry_adam.R")
# Loaded latest PIF artifacts, mortality results, previous YPLLs
pif2_input_bundle <- pif2_read_latest_artifact(
  directory = pif2_control_dir,
  stem = "aaf_engine_inputs_bundle",
  extension = "rds"
)
pif2_nested_bundle <- pif2_read_latest_artifact(
  directory = pif2_control_dir,
  stem = "aaf_nested_by_disease",
  extension = "rds"
)
pif2_mortality_results <- pif2_read_latest_artifact(
  directory = pif2_control_dir,
  pattern = "^Mortality Estimates WHO 2024\\.xlsx$|^Mortality Estimates.*\\.xlsx$",
  reader = readxl::read_xlsx
)
pif2_ypll_cache <- tryCatch(
  pif2_read_latest_artifact(
    directory = file.path(pif2_root_dir, "Mortalidad", "Matrices"),
    pattern = "^YPLL.*\\.rds$",
    reader = base::readRDS
  ),
  error = function(e) {
    pif2_message("[YPLL] No cached YPLL file loaded: %s", conditionMessage(e))
    NULL
  }
)
pif2_message("[bundle] Exposure input bundle: %s", basename(attr(pif2_input_bundle, "path")))
pif2_message("[bundle] Nested AAF bundle: %s", basename(attr(pif2_nested_bundle, "path")))
pif2_message("[bundle] Mortality table: %s", basename(attr(pif2_mortality_results, "path")))
if (!is.null(pif2_ypll_cache)) {
  pif2_message("[bundle] YPLL cache: %s", basename(attr(pif2_ypll_cache, "path")))
}
message(sprintf(
  "pif2-load-latest-artifacts elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[control-dir] Using: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control
[engine] Loaded aaf_unified.R and rr_registry_adam.R
[latest-file] Directory: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control
[latest-file] Pattern: ^aaf_engine_inputs_bundle_\d{8}\.rds$
[latest-file] Selection rule: embedded YYYYMMDD date, then modified time.
[latest-file] Candidate count: 3
[latest-file] Selected: aaf_engine_inputs_bundle_20260710.rds
[read-artifact] Reading: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control/aaf_engine_inputs_bundle_20260710.rds
[latest-file] Directory: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control
[latest-file] Pattern: ^aaf_nested_by_disease_\d{8}\.rds$
[latest-file] Selection rule: embedded YYYYMMDD date, then modified time.
[latest-file] Candidate count: 3
[latest-file] Selected: aaf_nested_by_disease_20260710.rds
[read-artifact] Reading: C:/Users/nDP/Desktop/ACC1240138_private/__andres_control/aaf_nested_by_disease_20260710.rds
[latest-file] 

Then, we loaded the registries for PIF scenarios.

In [8]:
#| label: pif2-expose-saved-objects
#| results: "hold"

.t0 <- Sys.time()
# The input bundle stores the exposure objects that were already built in
# expand_pif.ipynb. We expose them both as a named list and as ordinary objects
# because the existing engine functions expect names like g_fem_hed_list.
pif2_exposure_inputs <- pif2_input_bundle$exposure_inputs
base::list2env(pif2_exposure_inputs, envir = .GlobalEnv)
# Get the survey years
pif2_years <- as.integer(pif2_input_bundle$metadata$survey_years)
if (!length(pif2_years)) {
  pif2_years <- as.integer(names(pif2_exposure_inputs$g_fem_hed_list))
}
# Reuse the EXACT Monte Carlo settings and design closures the PAF run used, so
# the PIF is congruent with the validated PAF by construction:
#   * aaf_mc          -> list(n_sim, n_pca, seed, n_cores)
#   * aaf_uncertainty -> prev_method + neff/design_factor/neff_consumption/
#                        design_factor_consumption as function(year, group, sex)
#                        closures that read the survey design table (Kish n and
#                        PSU/region cluster factor). We do NOT rebuild them here.
pif2_aaf_mc          <- pif2_nested_bundle$inputs$aaf_mc
pif2_aaf_uncertainty <- pif2_nested_bundle$inputs$aaf_uncertainty
pif2_aaf_audit       <- pif2_nested_bundle$audits$aaf_adam_rr_audit
pif2_aaf_errors      <- pif2_nested_bundle$audits$aaf_adam_rr_errors

# ---------------------------------------------------------------------------
# Load the COMPLETE validated RR registry: ALL SIX scopes (not the former
# three-scope subset). These are the same RR-curve objects the PAF used, so the
# PIF can cover every one of the 23 pipeline diseases the AAF covers.
# ---------------------------------------------------------------------------
pif2_rr_registries <- list(
  cancer           = load_adam_rr_registry(scope = "cancer",   control_dir = pif2_control_dir, verbose = FALSE),
  hhd              = load_adam_rr_registry(scope = "hhd",      control_dir = pif2_control_dir, verbose = FALSE),
  general          = load_adam_rr_registry(scope = "general",  control_dir = pif2_control_dir, verbose = FALSE),
  ihd              = load_adam_rr_registry(scope = "ihd",      control_dir = pif2_control_dir, verbose = FALSE),
  ischaemic_stroke = load_adam_rr_registry(scope = "is",       control_dir = pif2_control_dir, verbose = FALSE),
  injuries         = load_adam_rr_registry(scope = "injuries", control_dir = pif2_control_dir, verbose = FALSE)
)

# Validate every registry BEFORE it is used: RR curves must be finite and
# non-negative, beta/covariance dimensions must match, covariance must be
# symmetric with a non-negative diagonal (this is validate_adam_rr_registry()).
pif2_registry_validation <- vapply(names(pif2_rr_registries), function(nm) {
  tryCatch({ validate_adam_rr_registry(pif2_rr_registries[[nm]]); "ok" },
           error = function(e) conditionMessage(e))
}, character(1))
if (any(pif2_registry_validation != "ok")) {
  stop("RR registry validation failed: ",
       paste(names(which(pif2_registry_validation != "ok")), collapse = ", "))
}

# Flat index for O(1) record lookup by (source_object, sex) for the non-age-banded
# families (cancer / hhd / general / injuries). CV (ihd / ischaemic_stroke) is
# age-banded and looked up separately by (sex, adam_age_band).
pif2_record_index <- local({
  idx <- list()
  for (scope in c("cancer", "hhd", "general", "injuries")) {
    for (rec in pif2_rr_registries[[scope]]) {
      idx[[paste(rec$source_object, rec$sex, sep = "::")]] <- rec
    }
  }
  idx
})

# ---------------------------------------------------------------------------
# Disease x sex x mode grid, DERIVED from the AAF audit (nb$audits$aaf_adam_rr_audit,
# 45 output tables). Driving the PIF from the PAF's own audit guarantees that the
# PIF covers EXACTLY the same output tables, modes (none/cap/explicit), and
# former-drinker settings the PAF used -- congruence by construction.
# ---------------------------------------------------------------------------
pif2_build_output_spec <- function(aaf_audit) {
  mode <- dplyr::recode(aaf_audit$hed_mode,
                        "none" = "nohed", "cap" = "cap", "explicit" = "explicit")
  tibble::tibble(
    output_name    = aaf_audit$output_name,
    disease        = aaf_audit$disease,            # pipeline label (e.g. "Other Pharyngeal Cancer")
    sex            = aaf_audit$sex,
    source_object  = aaf_audit$source_object,
    mode           = mode,                         # nohed | cap | explicit
    uses_hed       = mode %in% c("cap", "explicit"),
    age_banded     = mode == "cap",
    fd_uncertainty = as.logical(aaf_audit$fd_uncertainty),
    registry_scope = dplyr::case_when(
      mode == "explicit" ~ "injuries",
      mode == "cap" & grepl("Heart", aaf_audit$disease) ~ "ihd",
      mode == "cap" ~ "ischaemic_stroke",
      TRUE ~ NA_character_                         # nohed: resolved via pif2_record_index
    ),
    prefix = ifelse(aaf_audit$sex == "female", "Fem", "Male")
  )
}
pif2_output_spec <- pif2_build_output_spec(pif2_aaf_audit)

# Resolve the RR record for one spec row and one age group. CV records are chosen
# by Adam age band (15_64 scope: group 4 = 60-64 folds into the 35-64 band);
# every other family reuses one record across the four groups.
pif2_lookup_record <- function(spec_row, group, age_scope = "15_64") {
  if (isTRUE(spec_row$age_banded)) {
    band_map <- aaf_age_band_mapping(age_scope)
    band <- band_map$adam_age_band[band_map$group == group]
    return(.aaf_find_one(pif2_rr_registries[[spec_row$registry_scope]],
                         sex = spec_row$sex, adam_age_band = band))
  }
  rec <- pif2_record_index[[paste(spec_row$source_object, spec_row$sex, sep = "::")]]
  if (is.null(rec)) stop("No RR record for ", spec_row$source_object, " / ", spec_row$sex)
  rec$pipeline_disease <- spec_row$disease   # keep the AAF pipeline label (oral vs other-pharyngeal share one RR)
  rec
}

pif2_message("[objects] Exposed exposure objects: %s", paste(names(pif2_exposure_inputs), collapse = ", "))
pif2_message("[objects] Survey years: %s", paste(pif2_years, collapse = ", "))
pif2_message("[objects] Loaded + validated RR registries: %s", paste(names(pif2_rr_registries), collapse = ", "))
pif2_message("[objects] Registry record counts: %s",
             paste(names(pif2_rr_registries), vapply(pif2_rr_registries, length, integer(1)), sep = "=", collapse = ", "))
pif2_message("[objects] Output spec rows (PIF output tables): %s", nrow(pif2_output_spec))
pif2_message("[objects] Distinct pipeline diseases in spec: %s", dplyr::n_distinct(pif2_output_spec$disease))
pif2_message("[objects] Modes: %s",
             paste(names(table(pif2_output_spec$mode)), as.integer(table(pif2_output_spec$mode)), sep = "=", collapse = ", "))
pif2_message("[objects] MC settings reused from PAF: n_sim=%s n_pca=%s seed=%s",
             pif2_aaf_mc$n_sim, pif2_aaf_mc$n_pca, pif2_aaf_mc$seed)
pif2_message("[objects] Design closures reused: neff/design_factor are function(year,group,sex) = %s",
             is.function(pif2_aaf_uncertainty$neff))
message(sprintf(
  "pif2-expose-saved-objects elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


[objects] Exposed exposure objects: g_fem_list, g_male_list, g_fem_hed_list, g_male_hed_list, p_abs_list_fem, p_abs_list_male, p_form_list_fem, p_form_list_male, p_hed_list_fem, p_hed_list_male, x_vals, x_vals_nhed, x_vals_hed
[objects] Survey years: 2012, 2014, 2016, 2018, 2020, 2022, 2024
[objects] Loaded + validated RR registries: cancer, hhd, general, ihd, ischaemic_stroke, injuries
[objects] Registry record counts: cancer=15, hhd=2, general=16, ihd=6, ischaemic_stroke=6, injuries=6
[objects] Output spec rows (PIF output tables): 45
[objects] Distinct pipeline diseases in spec: 23
[objects] Modes: cap=4, explicit=6, nohed=35
[objects] MC settings reused from PAF: n_sim=10000 n_pca=1000 seed=2125
[objects] Design closures reused: neff/design_factor are function(year,group,sex) = TRUE
pif2-expose-saved-objects elapsed minutes: 0.00


## AAF long table from saved bundle

The nested bundle stores the AAF output tables by family. This cell reconstructs a long AAF table directly from that saved bundle, avoiding the old display tables and avoiding any raw-data rebuild.


In [9]:
#| label: pif2-aaf-long-from-bundle
#| results: "hold"

.t0 <- Sys.time()

pif2_wide_aaf_to_long <- function(wide_tbl, output_name = NA_character_) {
  # Convert one saved wide AAF table into the long schema used by downstream joins.
  # The saved tables use Fem1_point/Fem1_lower/Fem1_upper or Male1_* columns.
  wide_tbl <- tibble::as_tibble(wide_tbl)
  prefix <- if (any(grepl("^Fem\\d+_point$", names(wide_tbl)))) {
    "Fem"
  } else if (any(grepl("^Male\\d+_point$", names(wide_tbl)))) {
    "Male"
  } else {
    stop("Could not detect sex prefix in AAF table: ", output_name)
  }
# Format gender and age groups to numbers
  gender_value <- if (identical(prefix, "Fem")) "Mujer" else "Hombre"
  age_groups <- sort(as.integer(gsub(
    paste0("^", prefix, "(\\d+)_point$"),
    "\\1",
    grep(paste0("^", prefix, "\\d+_point$"), names(wide_tbl), value = TRUE)
  )))
# Format table
  purrr::map_dfr(age_groups, function(age_group_value) {
    tibble::tibble(
      year = wide_tbl$Year,
      age_group = age_group_value,
      gender = gender_value,
      disease = wide_tbl$disease,
      point = wide_tbl[[paste0(prefix, age_group_value, "_point")]],
      lower = wide_tbl[[paste0(prefix, age_group_value, "_lower")]],
      upper = wide_tbl[[paste0(prefix, age_group_value, "_upper")]],
      output_name = output_name
    )
  })
}
# function: wide to long
pif2_aaf_long_from_nested <- function(nested_bundle) {
  tables <- purrr::imap(
    nested_bundle$family_bundles,
    function(family_bundle, family_name) {
      purrr::imap_dfr(
        family_bundle$raw_result$tables,
        function(wide_tbl, output_name) {
          pif2_wide_aaf_to_long(wide_tbl, output_name = output_name) |>
            dplyr::mutate(family = family_name)
        }
      )
    }
  )
  dplyr::bind_rows(tables) |>
    dplyr::distinct(.data$year, .data$age_group, .data$gender, .data$disease, .data$output_name, .keep_all = TRUE) |>
    dplyr::arrange(.data$disease, .data$year, .data$age_group, .data$gender)
}
# deploy wide to long
pif2_aaf_long <- pif2_aaf_long_from_nested(pif2_nested_bundle)

pif2_message("[aaf-long] Rows: %s", nrow(pif2_aaf_long))
pif2_message("[aaf-long] Diseases: %s", dplyr::n_distinct(pif2_aaf_long$disease))
pif2_message("[aaf-long] Year range: %s-%s", min(pif2_aaf_long$year), max(pif2_aaf_long$year))

message(sprintf(
  "pif2-aaf-long-from-bundle elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

[aaf-long] Rows: 1260
[aaf-long] Diseases: 23
[aaf-long] Year range: 2012-2024
pif2-aaf-long-from-bundle elapsed minutes: 0.00


## PIF scenario grid

A **declarative scenario registry**. `shift_vol`/`shift_hed` are retained fractions (0.9 = 10% reduction), and all reductions are **relative** (recorded in `scale`). `engine_scenario` maps each row to the unified engine: `volume` reduces average consumption, `hed` reduces the HED fraction among current drinkers, and `both` combines them (the additive engine superset). Baseline is a no-change reference (PIF ≡ 0). `requires_hed` marks scenarios that only apply to HED-capable causes; applicability is decided per disease from the RR structure so a volume-only cause never shows a spurious HED effect.


In [10]:
#| label: pif2-scenario-grid
#| results: "hold"

.t0 <- Sys.time()

# Keep expensive simulation off by default. Turn this on only after the loading
# cells above complete and you want to test one small PIF run.
pif2_run_smoke_tests <- FALSE

# ---------------------------------------------------------------------------
# Declarative PIF scenario registry (Phase 5). One row per policy counterfactual.
# All reductions are RELATIVE (a fraction of the current level), recorded in
# `scale`. `shift_vol` / `shift_hed` are the RETAINED fractions (0.9 = -10%).
# engine_scenario maps to the unified engine:
#   "volume" -> reduce average consumption (everyone: x -> x*shift_vol)
#   "hed"    -> reduce HED prevalence (a fraction of binge drinkers become NHED)
#   "both"   -> combined volume + HED (the additive engine superset)
# Baseline is a volume scenario with shift 1 (no change) -> PIF == 0 by construction.
# `requires_hed` marks scenarios that only make sense for HED-capable diseases.
# ---------------------------------------------------------------------------
pif2_scenario_grid <- tibble::tribble(
  ~scenario_id,          ~scenario_label,                     ~scenario_family,     ~engine_scenario, ~volume_reduction_pct, ~hed_reduction_pct, ~shift_vol, ~shift_hed, ~requires_hed,
  "baseline",            "Baseline (no change)",              "Baseline",           "volume",         0,                     0,                  1.00,       1.00,       FALSE,
  "volume_reduction_10", "Average consumption -10%",          "Consumption volume", "volume",         10,                    0,                  0.90,       1.00,       FALSE,
  "volume_reduction_20", "Average consumption -20%",          "Consumption volume", "volume",         20,                    0,                  0.80,       1.00,       FALSE,
  "volume_reduction_30", "Average consumption -30%",          "Consumption volume", "volume",         30,                    0,                  0.70,       1.00,       FALSE,
  "hed_reduction_10",    "HED prevalence -10%",               "HED prevalence",     "hed",            0,                     10,                 1.00,       0.90,       TRUE,
  "hed_reduction_25",    "HED prevalence -25%",               "HED prevalence",     "hed",            0,                     25,                 1.00,       0.75,       TRUE,
  "hed_reduction_50",    "HED prevalence -50%",               "HED prevalence",     "hed",            0,                     50,                 1.00,       0.50,       TRUE,
  "combined_v10_h25",    "Combined: volume -10%, HED -25%",   "Combined",           "both",           10,                    25,                 0.90,       0.75,       TRUE,
  "combined_v20_h50",    "Combined: volume -20%, HED -50%",   "Combined",           "both",           20,                    50,                 0.80,       0.50,       TRUE,
  "combined_v30_h50",    "Combined: volume -30%, HED -50%",   "Combined",           "both",           30,                    50,                 0.70,       0.50,       TRUE
)
pif2_scenario_grid$scale <- "relative"   # reductions are relative fractions, not percentage points

# Applicability of a (disease, scenario) pair, decided from the RR structure:
# a HED/combined scenario is NOT applicable to a volume-only (no-HED) disease, so
# such cells are reported as NA with a machine-readable reason rather than a
# spurious zero effect.
pif2_scenario_applicability <- function(spec_row, scenario_row) {
  if (isTRUE(scenario_row$requires_hed) && !isTRUE(spec_row$uses_hed)) {
    return(list(applicable = FALSE,
                reason = "no HED RR component (volume-only cause); HED/combined scenario not applicable"))
  }
  list(applicable = TRUE, reason = NA_character_)
}

pif2_message("[scenarios] Defined %s scenarios (%s volume, %s HED, %s combined, 1 baseline).",
             nrow(pif2_scenario_grid),
             sum(pif2_scenario_grid$scenario_family == "Consumption volume"),
             sum(pif2_scenario_grid$scenario_family == "HED prevalence"),
             sum(pif2_scenario_grid$scenario_family == "Combined"))
pif2_message("[scenarios] Reduction scale: %s (retained fraction shift_vol/shift_hed).",
             unique(pif2_scenario_grid$scale))

message(sprintf(
  "pif2-scenario-grid elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


[scenarios] Defined 10 scenarios (3 volume, 3 HED, 3 combined, 1 baseline).
[scenarios] Reduction scale: relative (retained fraction shift_vol/shift_hed).
pif2-scenario-grid elapsed minutes: 0.00


## Per-cell PIF resolution

These helpers resolve one PIF cell's inputs **exactly as the validated AAF families do** — same gamma/prevalence pickers, same design closures — and build the `pif_confint()` argument list. Three modes are covered: `nohed` (cancer / HHD / general — volume scenarios only), `cap` (IHD / ischaemic stroke — J-curve HED, age-banded), and `explicit` (injuries — two-component HED with shared β₁). Multi-β RR uncertainty is drawn jointly from the covariance matrix inside the engine; former-drinker variance enters through `var_ln_rr_fd`.


In [11]:
#| label: pif2-pif-engine-wrapper
#| results: "hold"

.t0 <- Sys.time()

# ---------------------------------------------------------------------------
# Resolve one PIF cell's inputs EXACTLY as the AAF families do (same picker
# helpers, same gamma/prevalence objects), so the numerator and denominator of
# each PIF use the same population and age support as the validated PAF.
#   * nohed families (cancer/hhd/general): NHED gamma g_{sex}_list, no p_hed.
#   * cap/explicit families (ihd/is/injuries): HED-split gamma (nhed/hed) and p_hed.
# Returns list(ok=TRUE, gamma, gamma_hed, p_abs, p_form, p_hed) or ok=FALSE + reason.
# ---------------------------------------------------------------------------
pif2_resolve_cell_inputs <- function(spec_row, record, year, group, exposure) {
  sex <- spec_row$sex
  g_list      <- if (sex == "female") exposure$g_fem_list      else exposure$g_male_list
  g_hed_list  <- if (sex == "female") exposure$g_fem_hed_list  else exposure$g_male_hed_list
  p_abs_list  <- if (sex == "female") exposure$p_abs_list_fem  else exposure$p_abs_list_male
  p_form_list <- if (sex == "female") exposure$p_form_list_fem else exposure$p_form_list_male
  p_hed_list  <- if (sex == "female") exposure$p_hed_list_fem  else exposure$p_hed_list_male

  p_abs  <- .aaf_pick_prop(p_abs_list, year, group)
  p_form <- .aaf_pick_prop(p_form_list, year, group)

  if (identical(spec_row$mode, "nohed")) {
    gamma     <- g_list[[as.character(year)]][[group]]
    gamma_hed <- NULL; p_hed <- NULL
  } else {
    hed_entry  <- g_hed_list[[as.character(year)]][[group]]
    gamma      <- if (!is.null(hed_entry)) hed_entry$nhed else NULL
    gamma_hed  <- if (!is.null(hed_entry)) hed_entry$hed  else NULL
    year_index <- which(pif2_years == year)
    p_hed <- .aaf_prop_from_group_list(p_hed_list, year_index, group, year)
  }

  bad <- c(
    if (is.null(gamma)) "gamma_null",
    if (spec_row$uses_hed && is.null(gamma_hed)) "gamma_hed_null",
    if (is.na(p_abs)) "p_abs_NA",
    if (is.na(p_form)) "p_form_NA",
    if (spec_row$uses_hed && (is.null(p_hed) || is.na(p_hed))) "p_hed_NA"
  )
  if (length(bad)) return(list(ok = FALSE, reason = paste(bad, collapse = ",")))
  list(ok = TRUE, gamma = gamma, gamma_hed = gamma_hed,
       p_abs = p_abs, p_form = p_form, p_hed = p_hed)
}

# ---------------------------------------------------------------------------
# Build the pif_confint() argument list for one applicable cell + scenario. The
# design knobs (Kish n and cluster factor per survey question, plus the separate
# consumption-question design) are resolved from the SAME closures the PAF used
# (pif2_aaf_uncertainty), so the survey design is applied exactly once. Multi-beta
# RR uncertainty is carried by cov_beta (drawn jointly via mvrnorm inside the
# engine); former-drinker variance is carried by var_ln_rr_fd.
# ---------------------------------------------------------------------------
pif2_build_pif_args <- function(spec_row, record, inputs, scenario_row,
                                exposure, unc, mc, year, group, run_cfg) {
  sex <- spec_row$sex
  args <- list(
    gamma        = inputs$gamma,
    rr_fun       = record$RRCurrent,
    beta         = record$betaCurrent,
    p_abs        = inputs$p_abs,
    p_form       = inputs$p_form,
    ln_rr_fd     = record$lnRRFormer,
    var_ln_rr_fd = if (is.null(record$varLnRRFormer)) 0 else record$varLnRRFormer,
    fd_uncertainty = isTRUE(spec_row$fd_uncertainty),
    x            = exposure$x_vals,
    scenario     = scenario_row$engine_scenario,
    # Engine convention: `shift` is the retained fraction of the lever the scenario
    # acts on -> for "hed" it is the HED retained fraction (shift_hed); for
    # "volume"/"both" it is the volume retained fraction. `shift_hed` is read by
    # the engine only when scenario == "both".
    shift        = if (identical(scenario_row$engine_scenario, "hed")) scenario_row$shift_hed else scenario_row$shift_vol,
    shift_hed    = scenario_row$shift_hed,
    prev_method  = unc$prev_method,
    neff_prev                 = .aaf_resolve_cell(unc$neff, year, group, sex),
    design_factor             = .aaf_resolve_cell(unc$design_factor, year, group, sex),
    neff_consumption          = .aaf_resolve_cell(unc$neff_consumption, year, group, sex),
    design_factor_consumption = .aaf_resolve_cell(unc$design_factor_consumption, year, group, sex),
    n_sim = run_cfg$n_sim, n_pca = run_cfg$n_pca, seed = mc$seed,
    n_cores = run_cfg$n_cores, use_parallel = run_cfg$inner_parallel
  )
  if (spec_row$mode %in% c("nohed", "cap")) {
    args$cov_beta <- record$covBetaCurrent      # multi-beta joint draw (J-curve betas for CV)
    args$hed_mode <- "cap"
    if (spec_row$mode == "cap") { args$gamma_hed <- inputs$gamma_hed; args$p_hed <- inputs$p_hed }
  } else {  # explicit injuries: NHED beta1 shared, HED has its own beta2 + covariance
    args$cov_beta     <- NULL
    args$gamma_hed    <- inputs$gamma_hed
    args$p_hed        <- inputs$p_hed
    args$hed_mode     <- "explicit"
    args$rr_fun_hed   <- record$RRCurrent_binge
    args$beta_hed     <- record$betaCurrent_binge
    args$cov_beta_hed <- record$covBetaCurrent_binge
    args$share_beta1  <- TRUE
  }
  args
}

# Per-cell uncertainty-source flags (for the disease x scenario audit table).
pif2_uncertainty_flags <- function(spec_row, record) {
  ci_method <- .adam_ci_method(record)   # fixed | scalar | vcov (from the covariance rank)
  list(
    uses_volume_rr   = TRUE,                                   # every record exposes RRCurrent(x, beta)
    uses_hed_rr      = isTRUE(spec_row$uses_hed),
    uses_fd_rr       = is.finite(record$lnRRFormer) && abs(record$lnRRFormer) > 0,
    uses_beta_vcov   = identical(ci_method, "vcov") ||
                       (spec_row$mode == "explicit" && !is.null(record$covBetaCurrent_binge)),
    uses_weighted_gamma  = TRUE,                               # bundle 20260709: fit_gamma_weighted (see gamma audit)
    uses_design_variance = is.function(pif2_aaf_uncertainty$neff),
    ci_method = ci_method
  )
}

pif2_message("[pif-wrapper] Cell resolvers ready: pif2_resolve_cell_inputs / pif2_build_pif_args / pif2_uncertainty_flags.")

message(sprintf(
  "pif2-pif-engine-wrapper elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


[pif-wrapper] Cell resolvers ready: pif2_resolve_cell_inputs / pif2_build_pif_args / pif2_uncertainty_flags.
pif2-pif-engine-wrapper elapsed minutes: 0.00


## Run the PIF grid (all diseases × scenarios)

The orchestrator loops every output table × scenario × year × age group, resolves each cell's inputs exactly as the PAF families do, and calls `pif_confint()`. Non-applicable combinations (e.g. a HED scenario on a volume-only cancer) become `NA` rows with a machine-readable reason — never a spurious zero. `mode = "demo"` runs the latest wave at reduced `n_sim` so the notebook renders; set `pif2_pif_run_mode <- "full"` to reproduce the PAF's Monte Carlo depth over every wave. Results and the disease×scenario audit are saved with the dated project convention.


In [ ]:
#| label: pif2-run-grid
#| results: "hold"

.t0 <- Sys.time()
# ---------------------------------------------------------------------------
# Orchestrator (Phase 6). Loops output-table x scenario x year x age group.
# Non-applicable and invalid cells become NA rows carrying a machine-readable
# reason (never a spurious zero). Applicable cells are collected as fully-resolved
# jobs and run through pif_confint(). Common random numbers: every cell re-seeds
# its own L'Ecuyer streams from the fixed seed, so a scenario and its baseline share
# the same underlying draws for the same cell (the engine's serial==parallel
# invariant), which cancels Monte Carlo noise between comparable scenarios.
# Returns: long-format `results`, a disease x scenario `audit`, and the job count.
# ---------------------------------------------------------------------------
pif2_run_pif_grid <- function(spec, scenarios, exposure, unc, mc, years, groups, run_cfg) {
  long_rows  <- list(); jobs <- list(); job_meta <- list(); audit_rows <- list()
  na_row <- function(spec_row, scenario_row, year, group, applicable, reason, flags) {
    data.frame(
      disease = spec_row$disease, sex = spec_row$sex, age_group = group, year = year,
      output_name = spec_row$output_name,
      scenario_id = scenario_row$scenario_id, scenario_label = scenario_row$scenario_label,
      engine_scenario = scenario_row$engine_scenario,
      avg_consumption_change_pct = scenario_row$volume_reduction_pct,
      hed_prevalence_change_pct  = scenario_row$hed_reduction_pct,
      pif = NA_real_, pif_low = NA_real_, pif_up = NA_real_, n_used = NA_integer_,
      applicable = applicable, reason = reason,
      uses_volume_rr = flags$uses_volume_rr, uses_hed_rr = flags$uses_hed_rr,
      uses_fd_rr = flags$uses_fd_rr, uses_beta_vcov = flags$uses_beta_vcov,
      uses_weighted_gamma = flags$uses_weighted_gamma, uses_design_variance = flags$uses_design_variance,
      n_sim = run_cfg$n_sim, stringsAsFactors = FALSE)
  }

  for (si in seq_len(nrow(spec))) {
    spec_row <- spec[si, ]
    for (sc in seq_len(nrow(scenarios))) {
      scenario_row <- scenarios[sc, ]
      applic <- pif2_scenario_applicability(spec_row, scenario_row)
      for (year in years) for (group in groups) {
        record <- tryCatch(pif2_lookup_record(spec_row, group), error = function(e) NULL)
        flags  <- if (is.null(record))
          list(uses_volume_rr = NA, uses_hed_rr = spec_row$uses_hed, uses_fd_rr = NA,
               uses_beta_vcov = NA, uses_weighted_gamma = TRUE, uses_design_variance = is.function(unc$neff))
        else pif2_uncertainty_flags(spec_row, record)
        # one audit row per (disease x scenario), stamped on the first (year, group)
        if (year == years[[1]] && group == groups[[1]]) {
          audit_rows[[length(audit_rows) + 1L]] <- data.frame(
            disease_id = spec_row$output_name, disease = spec_row$disease, sex = spec_row$sex,
            scenario_id = scenario_row$scenario_id,
            uses_volume_rr = flags$uses_volume_rr, uses_hed_rr = flags$uses_hed_rr,
            uses_fd_rr = flags$uses_fd_rr, uses_beta_vcov = flags$uses_beta_vcov,
            uses_weighted_gamma = flags$uses_weighted_gamma, uses_design_variance = flags$uses_design_variance,
            age_support = "15-64 (edad_tramo 1-4)",
            applicable = applic$applicable, reason = applic$reason,
            warnings = NA_character_, stringsAsFactors = FALSE)
        }
        if (!applic$applicable) {
          long_rows[[length(long_rows) + 1L]] <- na_row(spec_row, scenario_row, year, group, FALSE, applic$reason, flags)
          next
        }
        if (is.null(record)) {
          long_rows[[length(long_rows) + 1L]] <- na_row(spec_row, scenario_row, year, group, TRUE, "record_lookup_failed", flags)
          next
        }
        inputs <- pif2_resolve_cell_inputs(spec_row, record, year, group, exposure)
        if (!isTRUE(inputs$ok)) {
          long_rows[[length(long_rows) + 1L]] <- na_row(spec_row, scenario_row, year, group, TRUE, paste0("invalid_inputs:", inputs$reason), flags)
          next
        }
        args <- pif2_build_pif_args(spec_row, record, inputs, scenario_row, exposure, unc, mc, year, group, run_cfg)
        jobs[[length(jobs) + 1L]] <- args
        job_meta[[length(job_meta) + 1L]] <- list(spec_row = spec_row, scenario_row = scenario_row,
                                                  year = year, group = group, flags = flags)
      }
    }
  }

  # ---- run all applicable jobs (each pif_confint controls its own inner MC) ----
  run_one <- function(args) tryCatch(do.call(pif_confint, args), error = function(e) e)
  results <- if (isTRUE(run_cfg$outer_parallel) && length(jobs) > 1L) {
    cl <- parallel::makeCluster(run_cfg$n_cores)
    on.exit(try(parallel::stopCluster(cl), silent = TRUE), add = TRUE)
    engine_file <- normalizePath(file.path(pif2_control_dir, "aaf_unified.R"), winslash = "/")
    parallel::clusterExport(cl, "engine_file", envir = environment())
    parallel::clusterEvalQ(cl, source(engine_file))
    parallel::clusterApplyLB(cl, lapply(jobs, function(a) utils::modifyList(a, list(use_parallel = FALSE, n_cores = 1L))), run_one)
  } else {
    lapply(jobs, run_one)
  }

  for (i in seq_along(results)) {
    res <- results[[i]]; m <- job_meta[[i]]
    if (inherits(res, "error")) {
      long_rows[[length(long_rows) + 1L]] <- na_row(m$spec_row, m$scenario_row, m$year, m$group, TRUE,
                                                    paste0("engine_error:", conditionMessage(res)), m$flags)
      next
    }
    long_rows[[length(long_rows) + 1L]] <- data.frame(
      disease = m$spec_row$disease, sex = m$spec_row$sex, age_group = m$group, year = m$year,
      output_name = m$spec_row$output_name,
      scenario_id = m$scenario_row$scenario_id, scenario_label = m$scenario_row$scenario_label,
      engine_scenario = m$scenario_row$engine_scenario,
      avg_consumption_change_pct = m$scenario_row$volume_reduction_pct,
      hed_prevalence_change_pct  = m$scenario_row$hed_reduction_pct,
      pif = res$point_estimate, pif_low = res$lower_ci, pif_up = res$upper_ci, n_used = res$n_used,
      applicable = TRUE, reason = NA_character_,
      uses_volume_rr = m$flags$uses_volume_rr, uses_hed_rr = m$flags$uses_hed_rr,
      uses_fd_rr = m$flags$uses_fd_rr, uses_beta_vcov = m$flags$uses_beta_vcov,
      uses_weighted_gamma = m$flags$uses_weighted_gamma, uses_design_variance = m$flags$uses_design_variance,
      n_sim = run_cfg$n_sim, stringsAsFactors = FALSE)
  }
  list(results = dplyr::bind_rows(long_rows), audit = dplyr::bind_rows(audit_rows), n_jobs = length(jobs))
}

# ---------------------------------------------------------------------------
# Run configuration. "demo" is a fast, render-friendly pass over the latest wave;
# "full" reproduces the PAF's Monte Carlo depth (n_sim from aaf_mc) over all waves.
# Both use the SAME estimator and the SAME design closures; only breadth/depth differ.
# ---------------------------------------------------------------------------
pif2_pif_run_mode <- "full"   # set to "full" for the complete multi-year, n_sim=10000 run
pif2_pif_run_config <- list(
  demo = list(years = as.integer(tail(pif2_years, 1L)), groups = 1:4,
              n_sim = 1000L, n_pca = 500L, n_cores = as.integer(pif2_aaf_mc$n_cores),
              inner_parallel = FALSE, outer_parallel = TRUE),
  full = list(years = pif2_years, groups = 1:4,
              n_sim = as.integer(pif2_aaf_mc$n_sim), n_pca = as.integer(pif2_aaf_mc$n_pca),
              n_cores = as.integer(pif2_aaf_mc$n_cores),
              inner_parallel = FALSE, outer_parallel = TRUE)
)
pif2_run_cfg <- pif2_pif_run_config[[pif2_pif_run_mode]]

pif2_pif_grid <- pif2_run_pif_grid(
  spec = pif2_output_spec, scenarios = pif2_scenario_grid,
  exposure = pif2_exposure_inputs, unc = pif2_aaf_uncertainty, mc = pif2_aaf_mc,
  years = pif2_run_cfg$years, groups = pif2_run_cfg$groups, run_cfg = pif2_run_cfg)
pif2_pif_results <- pif2_pif_grid$results
pif2_pif_audit   <- pif2_pif_grid$audit

# Attach the list of uncertainty components carried by each row (for provenance).
pif2_pif_results$uncertainty_components <- with(pif2_pif_results, paste(
  ifelse(uses_weighted_gamma, "weighted_gamma", NA),
  ifelse(uses_design_variance, "design_kish_cluster", NA),
  "dirichlet_prevalence",
  ifelse(uses_fd_rr, "fd_lognormal", NA),
  ifelse(uses_beta_vcov, "beta_mvrnorm", NA), sep = "+"))
pif2_pif_results$uncertainty_components <- gsub("(NA\\+)|(\\+NA)", "", pif2_pif_results$uncertainty_components)

# Save with the project's dated convention (does NOT overwrite the AAF bundles).
pif2_pif_stamp <- format(Sys.Date(), "%Y%m%d")
pif2_pif_results_path <- file.path(pif2_control_dir, paste0("pif2_pif_results_", pif2_pif_run_mode, "_", pif2_pif_stamp, ".rds"))
pif2_pif_audit_path   <- file.path(pif2_control_dir, paste0("pif2_pif_audit_",   pif2_pif_run_mode, "_", pif2_pif_stamp, ".rds"))
saveRDS(pif2_pif_results, pif2_pif_results_path)
saveRDS(pif2_pif_audit,   pif2_pif_audit_path)

pif2_message("[run-grid] mode=%s | years=%s | groups=%s | n_sim=%s",
             pif2_pif_run_mode, paste(pif2_run_cfg$years, collapse = ","),
             paste(pif2_run_cfg$groups, collapse = ","), pif2_run_cfg$n_sim)
pif2_message("[run-grid] jobs run: %s | result rows: %s | applicable: %s | non-applicable: %s",
             pif2_pif_grid$n_jobs, nrow(pif2_pif_results),
             sum(pif2_pif_results$applicable & !is.na(pif2_pif_results$pif)),
             sum(!pif2_pif_results$applicable))
pif2_message("[run-grid] diseases covered: %s / %s output tables",
             dplyr::n_distinct(pif2_pif_results$output_name[pif2_pif_results$applicable & !is.na(pif2_pif_results$pif)]),
             nrow(pif2_output_spec))
pif2_message("[run-grid] saved: %s", basename(pif2_pif_results_path))

message(sprintf(
  "pif2-run-grid elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))

~ 83 minutes

## Optional smoke test

This cell does not run unless `pif2_run_smoke_tests` is `TRUE`. It runs a single injuries cell through the SAME resolvers and engine the grid uses (volume, HED and combined scenarios) at the same simulation count, so it is sufficiently comparable.

In [ ]:
#| label: pif2-optional-smoke-test
#| results: "hold"

.t0 <- Sys.time()

# Optional single-cell wiring check using the SAME resolvers/engine the grid uses.
# It does not run unless pif2_run_smoke_tests is TRUE (set in pif2-scenario-grid).
if (isTRUE(pif2_run_smoke_tests)) {
  smoke_spec <- pif2_output_spec[pif2_output_spec$output_name == "ri_male", ]
  smoke_year <- as.integer(tail(pif2_years, 1L)); smoke_group <- 1L
  smoke_rec  <- pif2_lookup_record(smoke_spec, smoke_group)
  smoke_in   <- pif2_resolve_cell_inputs(smoke_spec, smoke_rec, smoke_year, smoke_group, pif2_exposure_inputs)
  smoke_cfg  <- list(n_sim = 10000L, n_pca = 1000L, n_cores = 12L, inner_parallel = FALSE, outer_parallel = FALSE)
  smoke_out  <- lapply(
    c("volume_reduction_10", "hed_reduction_10", "combined_v10_h25"),
    function(sid) {
      sr <- pif2_scenario_grid[pif2_scenario_grid$scenario_id == sid, ]
      args <- pif2_build_pif_args(smoke_spec, smoke_rec, smoke_in, sr,
                                  pif2_exposure_inputs, pif2_aaf_uncertainty, pif2_aaf_mc,
                                  smoke_year, smoke_group, smoke_cfg)
      res <- do.call(pif_confint, args)
      data.frame(scenario_id = sid, pif = res$point_estimate,
                 pif_low = res$lower_ci, pif_up = res$upper_ci)
    })
  pif2_smoke_pif <- do.call(rbind, smoke_out)
  pif2_message("[smoke] injuries MVA male %s group %s:", smoke_year, smoke_group)
  print(pif2_smoke_pif)
} else {
  pif2_message("[smoke] Not run. Set pif2_run_smoke_tests <- TRUE to enable.")
}

message(sprintf(
  "pif2-optional-smoke-test elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))



## YPLL bridge

This section uses a cached YPLL matrix when available. It joins the saved AAFs to YPLL first, then multiplies attributable YPLL by PIF to estimate avoidable YPLL.


In [ ]:
#| label: pif2-ypll-bridge
#| results: "hold"

.t0 <- Sys.time()

pif2_normalise_gender <- function(x) {
  dplyr::recode(
    as.character(x),
    "Men" = "Hombre",
    "Women" = "Mujer",
    "male" = "Hombre",
    "female" = "Mujer",
    .default = as.character(x)
  )
}

# Standardise the long PIF results (schema: disease, sex, year, age_group,
# scenario_id, engine_scenario, pif, pif_low, pif_up) into the join schema.
pif2_standardise_pif_table <- function(pif_tbl) {
  out <- tibble::as_tibble(pif_tbl)
  if (!"gender" %in% names(out) && "sex" %in% names(out)) {
    out$gender <- pif2_normalise_gender(out$sex)
  } else {
    out$gender <- pif2_normalise_gender(out$gender)
  }
  required_columns <- c(
    "year", "gender", "age_group", "disease", "scenario_id",
    "engine_scenario", "pif", "pif_low", "pif_up"
  )
  missing_columns <- setdiff(required_columns, names(out))
  if (length(missing_columns) > 0) {
    stop("PIF table is missing column(s): ", paste(missing_columns, collapse = ", "))
  }
  dplyr::select(out, dplyr::all_of(required_columns), dplyr::everything())
}

pif2_build_attributable_ypll <- function(ypll_cache, aaf_long) {
  if (is.null(ypll_cache)) {
    stop("No cached YPLL object is available. Add a YPLL cache or run the mortality/YPLL builder upstream.")
  }

  aaf_join <- aaf_long |>
    dplyr::select(dplyr::all_of(c("year", "age_group", "gender", "disease", "point", "lower", "upper")))

  ypll_cache |>
    tibble::as_tibble() |>
    dplyr::ungroup() |>
    dplyr::mutate(gender = pif2_normalise_gender(.data$gender)) |>
    dplyr::filter(.data$year %in% unique(aaf_join$year)) |>
    dplyr::left_join(
      aaf_join,
      by = c("year", "age_group", "gender", "disease")
    ) |>
    dplyr::mutate(
      ypll_att = .data$ypll * .data$point,
      ypll_att_low = .data$ypll * .data$lower,
      ypll_att_up = .data$ypll * .data$upper
    )
}

# Avoidable YPLL = attributable YPLL x PIF, per scenario. Only applicable PIF rows
# are joined (non-applicable disease x scenario cells carry NA and are dropped).
pif2_apply_pif_to_ypll <- function(attributable_ypll, pif_tbl) {
  pif_long <- pif2_standardise_pif_table(pif_tbl)
  pif_long <- pif_long[!is.na(pif_long$pif), , drop = FALSE]

  attributable_ypll |>
    dplyr::inner_join(
      pif_long,
      by = c("year", "gender", "age_group", "disease"),
      relationship = "many-to-many"
    ) |>
    dplyr::mutate(
      avoidable_ypll = .data$ypll_att * .data$pif,
      avoidable_ypll_low = .data$ypll_att * .data$pif_low,
      avoidable_ypll_up = .data$ypll_att * .data$pif_up
    )
}

pif2_attributable_ypll <- tryCatch(
  pif2_build_attributable_ypll(pif2_ypll_cache, pif2_aaf_long),
  error = function(e) {
    pif2_message("[YPLL] Attributable YPLL not built: %s", conditionMessage(e))
    NULL
  }
)

# Bridge PIF -> avoidable YPLL when both a YPLL cache and PIF results exist.
pif2_avoidable_ypll <- tryCatch(
  if (!is.null(pif2_attributable_ypll) && exists("pif2_pif_results")) {
    pif2_apply_pif_to_ypll(pif2_attributable_ypll, pif2_pif_results)
  } else NULL,
  error = function(e) {
    pif2_message("[YPLL] Avoidable YPLL not built: %s", conditionMessage(e))
    NULL
  }
)

if (!is.null(pif2_attributable_ypll)) {
  pif2_message("[YPLL] Attributable YPLL rows: %s", nrow(pif2_attributable_ypll))
  pif2_message("[YPLL] Missing AAF joins: %s", sum(is.na(pif2_attributable_ypll$point)))
}
if (!is.null(pif2_avoidable_ypll)) {
  pif2_message("[YPLL] Avoidable YPLL rows (by scenario): %s", nrow(pif2_avoidable_ypll))
  pif2_message("[YPLL] Scenarios in avoidable YPLL: %s",
               paste(sort(unique(pif2_avoidable_ypll$scenario_id)), collapse = ", "))
}

message(sprintf(
  "pif2-ypll-bridge elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


## Validation test harness (Phase 7)

Structural and grid-level checks over the computed PIF results and targeted single cells: baseline identity, HED/FD applicability, correct shift mapping, single application of the survey design (Kish n ÷ cluster factor, never doubled), joint covariance recovery, seed reproducibility, numerical validity, PIF ≤ 1 with negative values retained (not clipped), full disease coverage, **PAF congruence** (inputs reproduce the saved PAF), scenario isolation, and monotonicity only where the RR structure implies it. The cell stops loudly if any check fails.


In [ ]:
#| label: pif2-assertion-test-harness
#| results: "hold"

.t0 <- Sys.time()

# ---------------------------------------------------------------------------
# Phase 7 validation harness. Grid-level tests run on pif2_pif_results (the demo
# run); structural tests run targeted single cells at small n_sim so they are fast
# and deterministic. Every check is recorded; the cell stops loudly if any fail.
# ---------------------------------------------------------------------------
pif2_val <- list()
pif2_check <- function(id, cond, note = "") {
  pif2_val[[length(pif2_val) + 1L]] <<- data.frame(
    check = id, pass = isTRUE(cond), note = note, stringsAsFactors = FALSE)
  invisible(NULL)
}
pif2_aeq <- function(a, b, tol = 1e-9) all(is.finite(a) & is.finite(b) & abs(a - b) <= tol)
.exposure <- pif2_exposure_inputs; .unc <- pif2_aaf_uncertainty; .mc <- pif2_aaf_mc
.cfg <- list(n_sim = 300L, n_pca = 200L, n_cores = 1L, inner_parallel = FALSE, outer_parallel = FALSE)
.res <- pif2_pif_results
.appl <- .res[.res$applicable & !is.na(.res$pif), ]

# 1) Identity / zero-change: baseline (shift 1) -> PIF ~ 0
.base <- .res[.res$scenario_id == "baseline" & .res$applicable, ]
pif2_check("identity_baseline_zero", all(abs(.base$pif) < 1e-6, na.rm = TRUE),
           sprintf("max|pif|=%.1e", max(abs(.base$pif), na.rm = TRUE)))

# 3) Applicability: HED scenarios are non-applicable (NA) for volume-only diseases
.nohed <- pif2_output_spec$output_name[pif2_output_spec$mode == "nohed"]
.hn <- .res[.res$output_name %in% .nohed & .res$engine_scenario == "hed", ]
pif2_check("applicability_hed_on_nohed_NA",
           nrow(.hn) > 0 && all(!.hn$applicable) && all(is.na(.hn$pif)) && all(nzchar(na.omit(.hn$reason))),
           "HED scenarios NA + reason on volume-only causes")

# 3b) HED reduction actually lowers risk for HED-capable diseases (shift mapping)
.hc <- .appl[.appl$scenario_id == "hed_reduction_50", ]
pif2_check("hed_reduces_risk", nrow(.hc) > 0 && all(.hc$pif > 0),
           sprintf("min pif=%.4f over %d cells", suppressWarnings(min(.hc$pif)), nrow(.hc)))

# 4) FD applicability flags: injuries flagged uses_fd_rr = FALSE (RR_FD = 1)
pif2_check("fd_flag_injuries_false",
           isTRUE(all(!.res$uses_fd_rr[.res$output_name == "ri_male"])), "")

# 5) Age consistency + CV age-banding
pif2_check("age_support_subset", all(unique(.res$age_group) %in% 1:4), "")
.g1 <- pif2_lookup_record(pif2_output_spec[pif2_output_spec$output_name == "ihd_male", ], 1)
.g2 <- pif2_lookup_record(pif2_output_spec[pif2_output_spec$output_name == "ihd_male", ], 2)
pif2_check("cv_age_banding", .g1$adam_age_band == "15-34" && .g2$adam_age_band == "35-64", "")

# 6) Weighted gamma synthetic test (computed in the gamma-audit cell)
pif2_check("weighted_gamma_differs", isTRUE(pif2_gamma_weighted_differs),
           "weighted != unweighted under unequal weights")

# 7) Survey design applied ONCE (Kish n / cluster factor), not doubled
.np <- .aaf_resolve_cell(.unc$neff, 2022L, 2L, "male")
.df <- .aaf_resolve_cell(.unc$design_factor, 2022L, 2L, "male")
.ne <- .aaf_resolve_neff_eff(.np, .df)
.tbl <- environment(.unc$neff)$tbl
.row <- .tbl[.tbl$year == 2022 & .tbl$age_group == 2 & .tbl$sex == "male" & .tbl$variable == "abs", ]
pif2_check("design_applied_once",
           abs(.ne$abs - .row$neff_corr_engine) / .row$neff_corr_engine < 1e-4 &&
             abs(.ne$abs - .np / (.df$abs^2)) > 1,
           sprintf("engine=%.2f table=%.2f (double-apply=%.1f)", .ne$abs, .row$neff_corr_engine, .np / (.df$abs^2)))

# 8) Covariance: mvrnorm beta draws reproduce a supplied covariance
.covq <- matrix(c(0.010, 0.003, 0.003, 0.020), 2, 2); .bq <- c(0.5, -0.2)
set.seed(9); .draws <- t(replicate(30000, .aaf_draw_beta(.bq, .covq)))
pif2_check("covariance_recovered", max(abs(stats::cov(.draws) - .covq)) < 0.002,
           sprintf("max|emp-cov|=%.1e", max(abs(stats::cov(.draws) - .covq))))

# 9) Reproducibility: identical seed -> identical PIF summaries
.spec_liv <- pif2_output_spec[pif2_output_spec$output_name == "lican_male", ]
.rec_liv  <- pif2_lookup_record(.spec_liv, 2)
.in_liv   <- pif2_resolve_cell_inputs(.spec_liv, .rec_liv, 2022L, 2L, .exposure)
.args_liv <- pif2_build_pif_args(.spec_liv, .rec_liv, .in_liv,
                                 pif2_scenario_grid[pif2_scenario_grid$scenario_id == "volume_reduction_20", ],
                                 .exposure, .unc, .mc, 2022L, 2L, .cfg)
.ra <- do.call(pif_confint, .args_liv); .rb <- do.call(pif_confint, .args_liv)
pif2_check("reproducible_seed",
           pif2_aeq(.ra$point_estimate, .rb$point_estimate, 1e-12) &&
             pif2_aeq(.ra$lower_ci, .rb$lower_ci, 1e-12) && pif2_aeq(.ra$upper_ci, .rb$upper_ci, 1e-12), "")

# 10) Numerical validity: finite point + CI for applicable results
pif2_check("numerical_validity",
           all(is.finite(.appl$pif)) && all(is.finite(.appl$pif_low)) && all(is.finite(.appl$pif_up)), "")

# 11) Bounds: PIF <= 1; negative PIFs retained (reported, not clipped)
.neg <- sum(.appl$pif < -1e-9)
pif2_check("bounds_le_one", all(.appl$pif <= 1 + 1e-9) && all(.appl$pif_up <= 1 + 1e-9),
           sprintf("negative PIFs retained (flagged): %d", .neg))

# 12) All diseases: every AAF output table estimated under >= 1 applicable scenario
pif2_check("all_diseases_covered",
           setequal(unique(.appl$output_name), pif2_aaf_audit$output_name),
           sprintf("covered=%d / audit=%d", dplyr::n_distinct(.appl$output_name), nrow(pif2_aaf_audit)))

# 13) PAF congruence: resolved inputs reproduce the SAVED PAF for a cell
.aaf_args <- .args_liv[setdiff(names(.args_liv), c("scenario", "shift", "shift_hed"))]
.aaf_res  <- do.call(aaf_confint, .aaf_args)
.saved <- pif2_nested_bundle$family_bundles$cancer$raw_result$tables$lican_male
.saved_val <- .saved[.saved$Year == 2022, "Male2_point"]
pif2_check("paf_congruence", pif2_aeq(.aaf_res$point_estimate, .saved_val, 0.02),
           sprintf("my AAF=%.4f saved PAF=%.4f", .aaf_res$point_estimate, .saved_val))

# 14) Scenario isolation via the proven both-superset identities (real IHD cell)
.spec_ihd <- pif2_output_spec[pif2_output_spec$output_name == "ihd_male", ]
.rec_ihd  <- pif2_lookup_record(.spec_ihd, 2)
.in_ihd   <- pif2_resolve_cell_inputs(.spec_ihd, .rec_ihd, 2022L, 2L, .exposure)
.mk <- function(scn, sv, sh) do.call(pif_point, list(
  x = .exposure$x_vals, rr_nhed = .rec_ihd$RRCurrent, beta = .rec_ihd$betaCurrent,
  p_abs = .in_ihd$p_abs, p_form = .in_ihd$p_form, rr_fd = exp(.rec_ihd$lnRRFormer),
  y_nhed = .aaf_gamma_density(.exposure$x_vals, .aaf_gamma_pars(.in_ihd$gamma)),
  p_hed = .in_ihd$p_hed, rr_hed = "cap",
  y_hed = .aaf_gamma_density(.exposure$x_vals, .aaf_gamma_pars(.in_ihd$gamma_hed)),
  scenario = scn, shift = if (scn == "hed") sh else sv, shift_hed = sh))
pif2_check("isolation_volume_leaves_hed", pif2_aeq(.mk("both", 0.8, 1.0), .mk("volume", 0.8, 1.0)), "both(vol,1)==volume")
pif2_check("isolation_hed_leaves_volume", pif2_aeq(.mk("both", 1.0, 0.8), .mk("hed", 1.0, 0.8)), "both(1,hed)==hed")

# 15) Monotonicity ONLY for a monotone-increasing-RR cause (liver cancer)
.liv <- .appl[.appl$output_name == "lican_male" & .appl$engine_scenario == "volume", ]
.liv <- .liv[.liv$age_group == min(.liv$age_group), ]
.liv <- .liv[order(.liv$avg_consumption_change_pct), ]
pif2_check("monotone_increasing_rr", nrow(.liv) >= 2 && all(diff(.liv$pif) >= -1e-6),
           "not asserted for IHD/IS J-curve causes")

# ---- report ----
pif2_validation_report <- dplyr::bind_rows(pif2_val)
pif2_n_fail <- sum(!pif2_validation_report$pass)
print(knitr::kable(pif2_validation_report, caption = "Phase 7 validation harness"))
pif2_message("[validation] %d/%d checks passed.",
             sum(pif2_validation_report$pass), nrow(pif2_validation_report))
if (pif2_n_fail > 0L) {
  stop("Phase 7 validation FAILED: ",
       paste(pif2_validation_report$check[!pif2_validation_report$pass], collapse = ", "))
}

message(sprintf(
  "pif2-assertion-test-harness elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))


## Table 5 (PUC) IHD/IS sensitivity PIF

An **alternative RR source** for Ischaemic Heart Disease and Ischaemic Stroke: the Chile 2014 **Table 5 (PUC study)** sex-specific curves, instead of the WHO/Adam age-banded RRs used above. This reuses the *entire* PIF machinery (same orchestrator, scenario grid, survey-design closures, Monte Carlo settings) — only the RR curves change — and produces separate objects (`pif2_pif_results_table5`) plus a WHO-vs-Table 5 comparison. Table 5 RRs are **cardioprotective (J-curve)**, so volume-reduction PIFs can be negative (removing protection); these are retained and flagged, never clipped, and monotonicity is not imposed.


In [ ]:
#| label: pif2-table5-registries
#| results: "hold"

.t0 <- Sys.time()

# ---------------------------------------------------------------------------
# Table 5 (PUC, Chile 2014) sex-specific IHD/IS current-drinker RR curves -- an
# ALTERNATIVE RR source to the WHO/Adam age-banded RRs (GENERAL_ihd_RR_2018 /
# GENERAL_IS_RR_2018). These definitions are byte-identical to the canonical
# __andres_control/aaf_table5_ihd_is_experiment.R; replicated here so the notebook
# stays self-contained. Table 5 is NOT age-banded, so one RR per (disease, sex) is
# repeated across the three Adam age bands. The 'fact' column is stored as metadata
# and is NOT applied as a hidden x-scale (same decision as the Table 5 AAF
# experiment). IMPORTANT: these RRs are cardioprotective (J-curve) at low-to-moderate
# consumption, so under a VOLUME reduction the PIF can be NEGATIVE (removing
# protection). The engine retains negative PIFs; we flag them and do NOT clip or
# impose monotonicity on these causes.
# ---------------------------------------------------------------------------

# Table 5 ln(RR) forms (x = alcohol grams/day):
table5_rr_ihd_male   <- function(x, beta) exp(beta[[1L]] * sqrt(x) + beta[[2L]] * x^3)                 # B1*x^0.5 + B2*x^3
table5_rr_ihd_female <- function(x, beta) exp(beta[[1L]] * x + beta[[2L]] * x * log(x))                # B1*x + B2*x*ln(x)
table5_rr_is_male    <- function(x, beta) exp(beta[[1L]] * sqrt(x) + beta[[2L]] * sqrt(x) * log(x))    # B1*x^0.5 + B2*x^0.5*ln(x)
table5_rr_is_female  <- function(x, beta) exp(beta[[1L]] * sqrt(x) + beta[[2L]] * x)                   # B1*x^0.5 + B2*x

# Approximate log-normal former-drinker variance from a 95% CI.
table5_lognormal_var_from_ci <- function(lower, upper) ((log(upper) - log(lower)) / (2 * stats::qnorm(0.975)))^2

# Hard-coded Table 5 coefficients, SEs, former-drinker RRs (traceable to Table 5).
table5_rr_parameters <- function() data.frame(
  family = c("ihd", "ihd", "is", "is"),
  disease = c("Ischaemic Heart Disease", "Ischaemic Heart Disease", "Ischaemic Stroke", "Ischaemic Stroke"),
  sex = c("male", "female", "male", "female"),
  source_object = c("table5_ihd_male", "table5_ihd_female", "table5_is_male", "table5_is_female"),
  b1 = c(-0.046271, -0.052526, -0.141950, -0.249674), se_b1 = c(0.024037, 0.032510, 0.012866, 0.019163),
  b2 = c(0.000001, 0.014704, 0.039613, 0.037207),     se_b2 = c(0.000000, 0.007925, 0.001782, 0.000523),
  fact = c(1 / 3, 1 / 20, 1, 1), rr_former = c(1.25, 1.54, 0.97, 0.97),
  rr_former_lower = c(1.15, 1.17, 0.83, 0.83), rr_former_upper = c(1.36, 2.03, 1.14, 1.14),
  stringsAsFactors = FALSE)

# One RR record (compatible with compute_cv / the PIF cap mode). Diagonal beta
# covariance (independent betas, as supplied); former-drinker variance from the CI.
table5_make_record <- function(row, age_band) {
  rr_fun <- switch(paste(row$family, row$sex, sep = "_"),
                   ihd_male = table5_rr_ihd_male, ihd_female = table5_rr_ihd_female,
                   is_male = table5_rr_is_male,   is_female = table5_rr_is_female,
                   stop("Unsupported Table 5 RR row: ", row$family, " / ", row$sex))
  list(
    disease = row$disease, pipeline_disease = row$disease, rr_endpoint = row$disease,
    source_note = "Chile 2014 Table 5 (PUC) sex-specific RR; same RR across age bands (not age-banded); 'fact' not applied as x-scale.",
    pipeline_icd10 = if (identical(row$family, "ihd")) "I20-I25" else "G45-G46.8/I63/I65-I66/I67.2-I67.8/I69.3-I69.4",
    sex = row$sex, source_file = "Table 5, Chile 2014 (PUC); see aaf_table5_ihd_is_experiment.R",
    source_object = paste0(row$source_object, "_", gsub("[^0-9A-Za-z]+", "_", age_band)),
    adam_age_band = age_band, RRCurrent = rr_fun, betaCurrent = c(row$b1, row$b2),
    covBetaCurrent = diag(c(row$se_b1^2, row$se_b2^2), 2L),
    lnRRFormer = log(row$rr_former),
    varLnRRFormer = table5_lognormal_var_from_ci(row$rr_former_lower, row$rr_former_upper),
    table5_fact = row$fact, table5_rr_former_ci = c(row$rr_former_lower, row$rr_former_upper))
}

# Registry = 2 sexes x 3 age bands (same RR repeated per band, since Table 5 is not
# age-banded), so compute_cv / the cap-mode lookup work exactly as for WHO IHD/IS.
table5_make_registry <- function(family = c("ihd", "is")) {
  family <- match.arg(family)
  pars <- table5_rr_parameters(); pars <- pars[pars$family == family, , drop = FALSE]
  records <- list()
  for (i in seq_len(nrow(pars))) for (band in c("15-34", "35-64", "65+")) {
    rec <- table5_make_record(pars[i, , drop = FALSE], band)
    records[[paste(rec$sex, rec$source_object, sep = "::")]] <- rec
  }
  class(records) <- c("adam_rr_registry", "list")
  records
}

# Build, validate, and register the Table 5 registries as new scopes so the existing
# pif2_lookup_record()/pif2_run_pif_grid() machinery can use them unchanged. This is
# ADDITIVE: it does not touch the WHO-based pif2_output_spec or pif2_pif_results.
table5_ihd_registry <- table5_make_registry("ihd")
table5_is_registry  <- table5_make_registry("is")
validate_adam_rr_registry(table5_ihd_registry)
validate_adam_rr_registry(table5_is_registry)
pif2_rr_registries$table5_ihd <- table5_ihd_registry
pif2_rr_registries$table5_is  <- table5_is_registry

# Table 5 output spec (4 cap-mode tables), same schema pif2_run_pif_grid expects,
# plus an rr_source tag so results are never confused with the WHO run.
pif2_table5_output_spec <- tibble::tibble(
  output_name    = c("ihd_female", "ihd_male", "is_female", "is_male"),
  disease        = c("Ischaemic Heart Disease", "Ischaemic Heart Disease", "Ischaemic Stroke", "Ischaemic Stroke"),
  sex            = c("female", "male", "female", "male"),
  source_object  = c("table5_ihd_female", "table5_ihd_male", "table5_is_female", "table5_is_male"),
  mode           = "cap", uses_hed = TRUE, age_banded = TRUE, fd_uncertainty = TRUE,
  registry_scope = c("table5_ihd", "table5_ihd", "table5_is", "table5_is"),
  prefix         = c("Fem", "Male", "Fem", "Male"), rr_source = "table5_puc")

pif2_message("[table5] Registered scopes: table5_ihd (%d recs), table5_is (%d recs).",
             length(table5_ihd_registry), length(table5_is_registry))
pif2_message("[table5] Output spec: %s tables (IHD/IS x sex, cap mode).", nrow(pif2_table5_output_spec))
pif2_message("[table5] fact column (stored, NOT applied): %s",
             paste(sprintf("%s=%.4f", table5_rr_parameters()$source_object, table5_rr_parameters()$fact), collapse = ", "))

message(sprintf("pif2-table5-registries elapsed minutes: %.2f", pif2_elapsed_min(.t0)))

### Run + compare (WHO vs Table 5) + validate


In [ ]:
#| label: pif2-table5-run
#| results: "hold"

.t0 <- Sys.time()

# Run the Table 5 IHD/IS PIF with the SAME orchestrator, scenario grid, design
# closures and Monte Carlo settings as the main (WHO/Adam) run (pif2_run_cfg), so
# the two RR sources are directly comparable. Only the RR curves differ.
pif2_table5_grid <- pif2_run_pif_grid(
  spec = pif2_table5_output_spec, scenarios = pif2_scenario_grid,
  exposure = pif2_exposure_inputs, unc = pif2_aaf_uncertainty, mc = pif2_aaf_mc,
  years = pif2_run_cfg$years, groups = pif2_run_cfg$groups, run_cfg = pif2_run_cfg)
pif2_pif_results_table5 <- pif2_table5_grid$results
pif2_pif_audit_table5   <- pif2_table5_grid$audit
pif2_pif_results_table5$rr_source <- "table5_puc"

# Save with the dated convention (separate objects; the WHO run is untouched).
pif2_table5_results_path <- file.path(pif2_control_dir,
  paste0("pif2_pif_results_table5_", pif2_pif_run_mode, "_", format(Sys.Date(), "%Y%m%d"), ".rds"))
saveRDS(pif2_pif_results_table5, pif2_table5_results_path)

# ---- WHO (Adam) vs Table 5 (PUC) side-by-side for IHD/IS ----
pif2_cv_who <- pif2_pif_results[
  pif2_pif_results$disease %in% c("Ischaemic Heart Disease", "Ischaemic Stroke") &
    pif2_pif_results$applicable & !is.na(pif2_pif_results$pif),
  c("disease", "sex", "age_group", "year", "scenario_id", "pif", "pif_low", "pif_up")]
pif2_cv_table5 <- pif2_pif_results_table5[
  pif2_pif_results_table5$applicable & !is.na(pif2_pif_results_table5$pif),
  c("disease", "sex", "age_group", "year", "scenario_id", "pif", "pif_low", "pif_up")]
pif2_cv_compare <- dplyr::inner_join(
  pif2_cv_who, pif2_cv_table5,
  by = c("disease", "sex", "age_group", "year", "scenario_id"),
  suffix = c("_who", "_table5")) |>
  dplyr::mutate(pif_diff = .data$pif_table5 - .data$pif_who)

# ---- Mini validation harness for the Table 5 PIF ----
t5_val <- list()
t5_check <- function(id, cond, note = "") {
  t5_val[[length(t5_val) + 1L]] <<- data.frame(check = id, pass = isTRUE(cond), note = note, stringsAsFactors = FALSE)
}
.t5_appl <- pif2_pif_results_table5[pif2_pif_results_table5$applicable & !is.na(pif2_pif_results_table5$pif), ]
.t5_base <- pif2_pif_results_table5[pif2_pif_results_table5$scenario_id == "baseline" & pif2_pif_results_table5$applicable, ]
t5_check("identity_baseline_zero", all(abs(.t5_base$pif) < 1e-6, na.rm = TRUE))
t5_check("all_4_cv_tables_covered", setequal(unique(.t5_appl$output_name), pif2_table5_output_spec$output_name))
t5_check("hed_and_combined_computed", any(.t5_appl$engine_scenario == "hed") && any(.t5_appl$engine_scenario == "both"))
t5_check("bounds_le_one", all(.t5_appl$pif <= 1 + 1e-9) && all(.t5_appl$pif_up <= 1 + 1e-9))
t5_check("numerical_validity", all(is.finite(.t5_appl$pif)) && all(is.finite(.t5_appl$pif_low)) && all(is.finite(.t5_appl$pif_up)))

# PAF congruence: my aaf_confint on a Table 5 cell == compute_cv_aaf_from_registry.
# Use the FULL year vector for compute_cv so its positional p_hed index aligns with
# our which(pif2_years==year) index (both -> 2022).
.t5_cfg <- list(n_sim = 200L, n_pca = 200L, n_cores = 1L, inner_parallel = FALSE, outer_parallel = FALSE)
.t5_spec <- pif2_table5_output_spec[pif2_table5_output_spec$output_name == "ihd_female", ]
.t5_rec  <- pif2_lookup_record(.t5_spec, 2)
.t5_in   <- pif2_resolve_cell_inputs(.t5_spec, .t5_rec, 2022L, 2L, pif2_exposure_inputs)
.t5_args <- pif2_build_pif_args(.t5_spec, .t5_rec, .t5_in,
                                pif2_scenario_grid[pif2_scenario_grid$scenario_id == "volume_reduction_20", ],
                                pif2_exposure_inputs, pif2_aaf_uncertainty, pif2_aaf_mc, 2022L, 2L, .t5_cfg)
.t5_my_aaf <- do.call(aaf_confint, .t5_args[setdiff(names(.t5_args), c("scenario", "shift", "shift_hed"))])$point_estimate
.t5_cv <- compute_cv_aaf_from_registry(
  registry = table5_ihd_registry,
  g_fem_hed_list = pif2_exposure_inputs$g_fem_hed_list, g_male_hed_list = pif2_exposure_inputs$g_male_hed_list,
  p_abs_list_fem = pif2_exposure_inputs$p_abs_list_fem, p_abs_list_male = pif2_exposure_inputs$p_abs_list_male,
  p_form_list_fem = pif2_exposure_inputs$p_form_list_fem, p_form_list_male = pif2_exposure_inputs$p_form_list_male,
  p_hed_list_fem = pif2_exposure_inputs$p_hed_list_fem, p_hed_list_male = pif2_exposure_inputs$p_hed_list_male,
  x_vals = pif2_exposure_inputs$x_vals, years = pif2_years, age_groups = 1:4, age_scope = "15_64",
  prev_method = pif2_aaf_uncertainty$prev_method, neff = pif2_aaf_uncertainty$neff,
  design_factor = pif2_aaf_uncertainty$design_factor, neff_consumption = pif2_aaf_uncertainty$neff_consumption,
  design_factor_consumption = pif2_aaf_uncertainty$design_factor_consumption, fd_uncertainty = TRUE,
  n_sim = .t5_cfg$n_sim, n_pca = .t5_cfg$n_pca, seed = pif2_aaf_mc$seed,
  use_parallel = FALSE, target_output_names = "ihd_female")
.t5_cv_val <- .t5_cv$tables$ihd_female[.t5_cv$tables$ihd_female$Year == 2022, "Fem2_point"]
t5_check("paf_congruence", isTRUE(abs(.t5_my_aaf - .t5_cv_val) < 1e-9),
         sprintf("my AAF=%.5f compute_cv=%.5f", .t5_my_aaf, .t5_cv_val))

# Cardioprotective causes: negative PIFs are valid (removing protection). Reported,
# not clipped; monotonicity is intentionally NOT asserted here.
.t5_neg <- sum(.t5_appl$pif < -1e-9)

pif2_table5_validation <- dplyr::bind_rows(t5_val)
pif2_table5_n_fail <- sum(!pif2_table5_validation$pass)
print(knitr::kable(pif2_table5_validation, caption = "Table 5 (PUC) IHD/IS PIF validation"))
print(knitr::kable(utils::head(dplyr::arrange(pif2_cv_compare, .data$disease, .data$sex, .data$scenario_id), 12),
                   digits = 4, caption = "WHO (Adam) vs Table 5 (PUC) PIF for IHD/IS (same cells/scenarios)"))
pif2_message("[table5] %d/%d checks passed | negative PIFs retained (cardioprotection): %d | saved: %s",
             sum(pif2_table5_validation$pass), nrow(pif2_table5_validation), .t5_neg, basename(pif2_table5_results_path))
if (pif2_table5_n_fail > 0L) {
  stop("Table 5 PIF validation FAILED: ",
       paste(pif2_table5_validation$check[!pif2_table5_validation$pass], collapse = ", "))
}

message(sprintf("pif2-table5-run elapsed minutes: %.2f", pif2_elapsed_min(.t0)))


## Session info


In [ ]:

#| label: pif2-session-info
#| echo: true
#| results: "hold"

.t0 <- Sys.time()

message(paste0("R library: ", Sys.getenv("R_LIBS_USER")))
message(paste0("Date: ", withr::with_locale(new = c("LC_TIME" = "C"), code = Sys.time())))
message(paste0("Editor context: ", getwd()))
utils::sessionInfo()

message(sprintf(
  "pif2-session-info elapsed minutes: %.2f",
  pif2_elapsed_min(.t0)
))
